# 02 LST Retrieval and Exploration

            This notebook guides Landsat 8/9 Collection 2 Level-2 preparation and LST computation.
            It does not force Earth Engine authentication or downloads; it previews commands and validates
            local files before processing.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("LST Retrieval Exploration", YEAR)

## Step 1 - Understand the required Landsat bands

For LST, the pipeline needs `ST_B10`. For later indices, it also needs `SR_B2`, `SR_B3`, `SR_B4`, `SR_B5`, and `SR_B6`.

In [ ]:
landsat_files = find_files("data/raw/landsat", [f"*{YEAR}*.tif", f"*{YEAR}*.TIF"])
            print(f"Found {len(landsat_files)} Landsat GeoTIFF file(s) for {YEAR}.")
            for path in landsat_files[:30]:
                print(path.relative_to(PROJECT_ROOT))

## Step 2 - Preview Earth Engine export commands

Run authentication once, then export dry-season composites. Keep `RUN_COMMANDS = False` until you are ready.

In [ ]:
run_command(["python", "scripts/02a_gee_export_landsat.py", "--auth-only"], dry_run=True)
            run_command(["python", "scripts/02a_gee_export_landsat.py", "--years", "2015", "2020", "2023"], dry_run=True)

## Step 3 - Validate local Landsat inputs

In [ ]:
run_command(["python", "scripts/02_download_or_prepare_satellite_data.py", "--year", YEAR], dry_run=not RUN_COMMANDS)

## Step 4 - Compute LST

Formula: `lst_kelvin = thermal_band * 0.00341802 + 149.0`; `lst_celsius = lst_kelvin - 273.15`.

In [ ]:
run_command(["python", "scripts/03_compute_lst.py", "--year", YEAR], dry_run=not RUN_COMMANDS)

## Step 5 - Inspect the LST raster metadata

In [ ]:
lst_path = project_path(f"data/processed/lst/lst_ibadan_{YEAR}_celsius.tif")
            if lst_path.exists():
                print(raster_info(lst_path))
                print(raster_stats(lst_path))
            else:
                print("LST raster not found yet.")

## Step 6 - Plot the LST raster when available

In [ ]:
plot_raster(lst_path, f"Ibadan Land Surface Temperature {YEAR}", cmap="inferno")